# 01 — EDA: Utah FORGE 56-32 Geothermal Drilling Sensors (1-sec)

**Goal (paper §II, Data Understanding):** descriptive stats (mean/std/skew), scale disparity audit, correlation matrix, outlier/boxplot, missing-code audit (`-999.25`).
Output is English-ready for IEEE tables/figures.

**Kaggle setup:** Add private dataset `utah-forge-56-32-1sec` as Notebook Input. No GPU needed (CPU).

In [ ]:
import os
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns

SENSOR_8 = [
    'Rate Of Penetration (ft_per_hr)', 'Weight on Bit (klbs)',
    'Rotary RPM (RPM)', 'Standpipe Pressure (psi)',
    'Rotary Torque (kft_lb)', 'Hook Load (klbs)',
    'Total Pump Output (gal_per_min)', 'Differential Pressure (psi)',
]
CSV_NAME = '56-32 1sec data 27029986.csv'
SLUG = 'utah-forge-56-32-1sec'
NROWS, SEED = 100_000, 42

def resolve_input(csv=CSV_NAME, slug=SLUG):
    root = Path('/kaggle/input')
    if root.exists():
        hits = list(root.rglob(csv))
        if hits:
            return hits[0]
        for p in root.rglob('*.csv'):
            if '56-32' in p.name or '1sec' in p.name or '27029986' in p.name:
                return p
        csvs = [p for p in root.rglob('*.csv') if p.is_file()]
        if csvs:
            return max(csvs, key=lambda p: p.stat().st_size)
        pqs = [p for p in root.rglob('*.parquet') if p.is_file()]
        if pqs:
            return max(pqs, key=lambda p: p.stat().st_size)
    for c in [Path('dataset')/csv, Path('../dataset')/csv, Path.cwd()/'dataset'/csv]:
        if c.exists(): return c
    listing = [str(p) for p in sorted(root.rglob('*'))][:30] if root.exists() else ['</kaggle/input not found>']
    raise FileNotFoundError('Dataset CSV not found. /kaggle/input contains: ' + str(listing) + ' | Fix: right panel Add Input -> add your private dataset -> re-run.')

if Path('/kaggle/working').exists(): OUT = Path('/kaggle/working')  # live session: visible in file browser, downloadable, saved on Save Version
elif Path('/kaggle/output').exists(): OUT = Path('/kaggle/output')
else: OUT = Path('../figures')
OUT.mkdir(parents=True, exist_ok=True)
print('input:', resolve_input())
print('output:', OUT)

In [ ]:
path = resolve_input()
usecols = ['YYYY/MM/DD','HH:MM:SS'] + SENSOR_8
df = pd.read_csv(path, usecols=usecols, nrows=NROWS*2, low_memory=True)
rng = np.random.RandomState(SEED)
if len(df) > NROWS:
    df = df.iloc[np.sort(rng.choice(len(df), NROWS, replace=False))].reset_index(drop=True)
for c in SENSOR_8:
    df[c] = pd.to_numeric(df[c], errors='coerce').astype('float32')
df['datetime'] = pd.to_datetime(df['YYYY/MM/DD']+' '+df['HH:MM:SS'], format='%Y/%m/%d %H:%M:%S', errors='coerce')
print(df.shape)
display(df.head(3))
print('time range:', df['datetime'].min(), '->', df['datetime'].max())

In [ ]:
desc = df[SENSOR_8].describe(percentiles=[.25,.5,.75]).T
desc['skew'] = df[SENSOR_8].skew(numeric_only=True)
desc['range'] = desc['max'] - desc['min']
display(desc.round(2))
desc.round(2).to_csv(OUT/'table1_descriptive.csv')
print('saved table1_descriptive.csv')
# Scale disparity check: max/std ratio highlights Euclidean bias risk

In [ ]:
full = pd.read_csv(resolve_input(), usecols=['Pason Gas (percent)','Gamma (api)'], nrows=20000)
print('Pason Gas == -999.25:', (full['Pason Gas (percent)']==-999.25).mean())
print('Gamma == -999.25:', (full['Gamma (api)']==-999.25).mean())
print('=> DROP both columns (inactive sensors).')
print('NaN in 8 sensors:', df[SENSOR_8].isna().sum().to_dict())

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(df[SENSOR_8].corr(numeric_only=True), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Matrix — 8 Geothermal Drilling Sensors')
plt.tight_layout(); plt.savefig(OUT/'fig1_correlation.png', dpi=150); plt.show()

In [ ]:
fig, ax = plt.subplots(2, 4, figsize=(14,6), sharey=False)
for a, c in zip(ax.ravel(), SENSOR_8):
    v = df[c].dropna().to_numpy()
    lo, hi = np.percentile(v, [1, 99])  # clip 1-99pct for readable boxplot
    a.boxplot(v[(v>=lo)&(v<=hi)], vert=True)
    a.set_title(c.split(' (')[0], fontsize=9)
plt.suptitle('Boxplots (1st–99th pct clipped) — Outlier Detection')
plt.tight_layout(); plt.savefig(OUT/'fig2_boxplots.png', dpi=150); plt.show()

#### Cell: Export & verify — collect everything for download
**What:** lists what was actually written to `OUT`, copies it into `/kaggle/working` (the live, downloadable folder), and packs it into one `kaggle_outputs.zip`.
**Why:** if this cell reports `<EMPTY>`, the cells above never ran (error or skipped) — outputs cannot exist. The zip gives you one-click download instead of per-file clicks.
**Read this:** the printed file list (expect `table*.csv` + `fig*.png`) and the zip size; then download via the file browser.


In [ ]:
from pathlib import Path
import shutil, zipfile
OUT.mkdir(parents=True, exist_ok=True)
live = Path('/kaggle/working') if Path('/kaggle/working').exists() else OUT
moved = []
for f in sorted(OUT.glob('*')):
    if not f.is_file() or f.suffix == '.zip':
        continue
    t = live / f.name
    if f.resolve() != t.resolve():
        shutil.copy2(f, t)
    moved.append(f.name)
print('OUT =', OUT)
print('files:', moved if moved else '<EMPTY — cells above did not run or errored, re-run from the top>')
zp = live / 'kaggle_outputs.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for name in moved:
        z.write(live / name, name)
print(f'zip: {zp} ({zp.stat().st_size/1024:.1f} KB)')
print('Download: file browser ->', live, '-> kaggle_outputs.zip. For the paper link: Save Version -> Output tab.')

try:
    from IPython.display import FileLink, display as _display
    _display(FileLink(str(zp)))
    print('Click the link above to download the zip directly (no file browser needed).')
except Exception as _e:
    print('direct-link unavailable:', _e)
